# Kalshi sports trends: run the analysis

This notebook re-runs every result in the project in about a minute. It uses the data already saved in the repo, so there's nothing to download from Kalshi and no API key needed.

**Run everything:** Runtime → Run all.

Live tool: https://adiamlani130.github.io/KalshiArbitrageBettor/

In [ ]:
# Get the project (code + data) from GitHub
!git clone -q https://github.com/adiamlani130/KalshiArbitrageBettor.git
%cd KalshiArbitrageBettor

## Tennis: betting on or against the ATP set-1 winner
Every bet is $10 at the real asking price 1–2 minutes after set 1 ends, minus Kalshi's fee.

In [ ]:
!python analysis/tennis_analysis.py

## NBA: comebacks, leaders, favorites and back-to-backs
Matches every Kalshi game market to ESPN play-by-play, then prices each bet at the moment it happens.

In [ ]:
!python analysis/nba_build.py
!python analysis/nba_analysis.py

## Reproduce the original Colab run
The first data pull was smaller (it couldn't read Kalshi's archive yet). These commands reproduce its numbers exactly, including the +15.3% tennis result.

In [ ]:
!python analysis/tennis_analysis.py data/original_run results/original_run
!python analysis/nba_build.py data/original_run results/original_run
!python analysis/nba_analysis.py data/original_run results/original_run

## Try your own tennis strategy
Change the settings and re-run the cell. `SIDE` is who you bet on right after set 1; the price band is the **set-1 winner's** price at that moment (0.70 = 70¢ = −233).

In [ ]:
import sys, pandas as pd
sys.path.insert(0, "analysis")
from common import summarize

SIDE = "loser"            # "winner" = back the set-1 winner, "loser" = bet against them
PRICE_MIN, PRICE_MAX = 0.70, 0.80
FROM, TO = "2026-01-01", "2026-12-31"

t = pd.read_csv("data/tennis/set1_all.csv")
t["date"] = pd.to_datetime(t.code.str[:7], format="%y%b%d").dt.strftime("%Y-%m-%d")
slams = [("2026-01-18", "2026-02-01"), ("2026-05-24", "2026-06-07"), ("2026-06-29", "2026-07-12"), ("2026-08-31", "2026-09-13")]
t = t[(t.ask > 0.02) & (t.ask < 0.99) & t.bid.notna() & ~t.date.apply(lambda d: any(a <= d <= b for a, b in slams))]
x = t[t.ask.between(PRICE_MIN, PRICE_MAX - 1e-9) & t.date.between(FROM, TO)]
won = x.won.astype(bool)
price, win = (x.ask, won) if SIDE == "winner" else (1 - x.bid, ~won)
pd.DataFrame([summarize(f"Bet the set-1 {SIDE}", price, win, x.date, "2026-07-15")])